In [0]:
%sql
Create schema if not exists workspace.bronze;

In [0]:
# Para o csv de customers. Na celulas seguintes eu vou ler o csv colocar a coluna de time ingestion e salvar na camada bronze os dados do jeito que estao. Nessa eu vou fazer com o csv de customers
from pyspark.sql.functions import current_timestamp

caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/olist_customers_dataset.csv"

df_olist_costumers = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(caminho)
)

df_bronzee = df_olist_costumers.withColumn("timestamp_ingestion", current_timestamp())

df_bronzee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_customers")

In [0]:
#para a tabela de geolocalization

caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/olist_geolocation_dataset.csv"

df_olist_geo = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(caminho)
)

df_bronze = df_olist_geo.withColumn("timestamp_ingestion", current_timestamp())

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_geolocalizacao")

In [0]:
# para a tabela de ordem de itens
caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/olist_order_items_dataset.csv"

df_olist_itens = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(caminho)
)

df_bronze = df_olist_itens.withColumn("timestamp_ingestion", current_timestamp())

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_order_items")

In [0]:
# para a tabela de pagamentos

caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/olist_order_payments_dataset.csv"

df_pagamentos = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferschema", True)
    .load(caminho)
)

df_bronze = df_pagamentos.withColumn("timestamp_ingestion",current_timestamp())

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_order_payments") 

In [0]:
#para a tabela de reviews

caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/olist_order_reviews_dataset.csv"

df_reviews = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferschema", True)
    .load(caminho)
)

df_bronze = df_reviews.withColumn("timestamp_ingestion", current_timestamp())

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_order_reviews")

In [0]:
# para a tabela de pedidos

caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/olist_orders_dataset.csv"

df_orders = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferschema", True)
    .load(caminho)
)

df_bronze = df_orders.withColumn("timestamp_ingestion", current_timestamp())

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_orders")

In [0]:
#tabela produtos

caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/olist_products_dataset.csv"

df_products = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferschema", True)
    .load(caminho)
)

df_bronzee = df_products.withColumn("timestamp_ingestion", current_timestamp())

df_bronzee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_products")

In [0]:
#tabela vendedores

caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/olist_sellers_dataset.csv"

df_sellers = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferschema", True)
    .load(caminho)
)

df_bronze = df_sellers.withColumn("timestamp_ingestion", current_timestamp())

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_sellers")

In [0]:
# tabela produto categoria

caminho = "/Volumes/workspace/default/vol_olist2/Base de Dados da Atividade-20260404T181946Z-1-001/Base de Dados da Atividade/product_category_name_translation.csv"

df_product_category = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferschema", True)
    .load(caminho)
)

df_bronze = df_product_category.withColumn("timestamp_ingestion", current_timestamp())

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.bronze.tb_product_category_name_translation")

OBSERVAÇÃO: botei a linha de codigo .option("overwriteSchema", "true") pois eu tinha baixado o arquivo de codigo errado e ja tinha criado as tabelas, entao fiz isso para que elas sobrescrevessem.

A partir de agora vamos fazer a ingestao de api do BACEN para conseguir pegar a contação do dolar em um intervalo de tempo


In [0]:
from pyspark.sql import functions as F

limites = (
    spark.table("workspace.bronze.tb_orders")
    .select(
            F.min("order_purchase_timestamp").alias("min"),
            F.max("order_purchase_timestamp").alias("max")
        ).collect()[0]
    )

data_inicio_formatada = limites["min"].strftime("%m-%d-%Y")
data_fim_formatada = limites["max"].strftime("%m-%d-%Y")

# Esse codigo eu so fiz pegar o menor e o maior timestamp da tabela de pedidos e tratar eles para que fiquem no formato de data pedido

In [0]:
dbutils.widgets.text("data_inicio", data_inicio_formatada)
dbutils.widgets.text("data_final", data_fim_formatada)

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_final")
# nessa celula eu criei os widgets. Apos isso, vamos nos conectar com a API do BACEN e retirar as informações necessarias

In [0]:
#coloco a api de cotação do dolar no meu projeto 
import json
from urllib.request import urlopen

URL = ("https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&"
    f"@dataFinalCotacao='{data_fim}'&"
    "$select=dataHoraCotacao,cotacaoCompra&"
    "$format=json")
     
with urlopen(URL) as response:
    payload = json.loads(response.read().decode("utf-8"))

registros = payload["value"]

print("quantidade de resgistros é: ", len(registros))
print(registros[:3])

quantidade de resgistros é:  530
[{'cotacaoCompra': 3.2715, 'dataHoraCotacao': '2016-09-05 13:09:55.659'}, {'cotacaoCompra': 3.2446, 'dataHoraCotacao': '2016-09-06 13:02:39.984'}, {'cotacaoCompra': 3.1928, 'dataHoraCotacao': '2016-09-08 13:03:53.968'}]


In [0]:
from pyspark.sql.functions import current_timestamp, col, to_timestamp

df_cotacao = spark.createDataFrame(registros)

df_cotacao = (
    df_cotacao.withColumn("dataHoraCotacao", to_timestamp(col("dataHoraCotacao")))
    .withColumn("timestamp_ingestion", current_timestamp())
)

display(df_cotacao)
df_cotacao.printSchema()

cotacaoCompra,dataHoraCotacao,timestamp_ingestion
3.2715,2016-09-05T13:09:55.659Z,2026-04-07T12:18:54.908Z
3.2446,2016-09-06T13:02:39.984Z,2026-04-07T12:18:54.908Z
3.1928,2016-09-08T13:03:53.968Z,2026-04-07T12:18:54.908Z
3.2632,2016-09-09T13:14:00.885Z,2026-04-07T12:18:54.908Z
3.2848,2016-09-12T13:08:01.541Z,2026-04-07T12:18:54.908Z
3.2966,2016-09-13T13:03:56.534Z,2026-04-07T12:18:54.908Z
3.3256,2016-09-14T13:05:51.819Z,2026-04-07T12:18:54.908Z
3.332,2016-09-15T13:08:34.825Z,2026-04-07T12:18:54.908Z
3.2998,2016-09-16T13:04:11.365Z,2026-04-07T12:18:54.908Z
3.263,2016-09-19T13:07:42.039Z,2026-04-07T12:18:54.908Z


root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: timestamp (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)



In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

df_cotacao.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.tb_cotacao_dolar")
